#Reading From Bronze Table

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

## erp_px_cat_g1v2 Transformation

In [0]:
# ============================================================
# ERP_PX_CAT_G1V2 — Product Category Reference
# ============================================================
df_cat = spark.table("dev_project.bronze.erp_px_cat_g1v2")

# Trim all string columns
string_cols = [f.name for f in df_cat.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    df_cat = df_cat.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None)
         .otherwise(F.trim(F.col(col_name)))
    )

# Rename + final select
df_cat_silver = df_cat.select(
    F.col("ID").alias("category_id"),
    F.col("CAT").alias("category"),
    F.col("SUBCAT").alias("subcategory"),
    F.col("MAINTENANCE").alias("maintenance")
)

# Validation
print("\n" + "="*50)
print(" ERP_PX_CAT_G1V2 VALIDATION")
print("="*50)
print(f"Total records:      {df_cat_silver.count():,}")
print(f"Null category IDs:  {df_cat_silver.filter(F.col('category_id').isNull()).count()}")
print(f"Category breakdown:")
df_cat_silver.groupBy("category").count().show()
print(f"Maintenance breakdown:")
df_cat_silver.groupBy("maintenance").count().show()

# Write — overwrite (reference table, current state only)
(
    df_cat_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("dev_project.silver.erp_px_cat_g1v2")
)
print("erp_px_cat_g1v2 Silver write complete.")

In [0]:
df_prd = spark.table("dev_project.silver.crm_prd_info") \
    .withColumn("parts", F.split(F.col("product_key"), "-")) \
    .withColumn("num_parts", F.size(F.col("parts"))) \
    .withColumn(
        "prd_key_short",
        F.when(
            F.col("num_parts") == 5,
            F.concat_ws("-",
                F.get(F.col("parts"), 2),
                F.get(F.col("parts"), 3),
                F.get(F.col("parts"), 4)
            )
        ).when(
            F.col("num_parts") == 4,
            F.concat_ws("-",
                F.get(F.col("parts"), 2),
                F.get(F.col("parts"), 3)
            )
        )
    ).select("product_key", "num_parts", "prd_key_short").distinct()

df_sales = spark.table("dev_project.silver.crm_sales_details") \
    .select("product_key").distinct()

matched = df_sales.join(df_prd, df_sales.product_key == df_prd.prd_key_short, how="inner").count()
unmatched = df_sales.join(df_prd, df_sales.product_key == df_prd.prd_key_short, how="left_anti").count()

print(f"Matched:   {matched:,}")
print(f"Unmatched: {unmatched:,}")